<div class="alert alert-block alert-info">
Before you start running this notebook, create a cloud app in Verily Workbench with a <b>low compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>JupyterLab</b>.</li>
    <li>On the <b>Compute options</b> step, the default profile (2 CPUs, 13 GB of memory) is sufficient.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook summarizes haplotype-resolved Hifiasm assembly metrics for Phase 1 long-read samples and walks through an example read-to-contigs alignment. It aggregates contig counts and N50 from `report_map.txt` files, then maps PacBio reads from an example sample to its haplotigs with minimap2.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Example of one sample's hifiasm assembly report.
!gsutil -u $GOOGLE_PROJECT cat gs://fc-aou-datasets-controlled/pooled/longreads/v7_base/hifiasm/example_sample/report_map.txt | column -t

In [ ]:
# Concatenate every sample's assembly report into one file (the Controlled Tier bucket is requester-pays).
!gsutil -u $GOOGLE_PROJECT cat gs://fc-aou-datasets-controlled/pooled/longreads/v7_base/hifiasm/*/report_map.txt > full_report_map.txt

In [ ]:
# Number of contigs per haplotype (column 3 is haplotype 1, column 4 is haplotype 2).
# Rows containing 'gt_' (thresholded contig counts) are excluded.
hap1_num = !cat full_report_map.txt | grep '#_contigs' | grep -v 'gt_' | awk '{ print $3 }'
hap2_num = !cat full_report_map.txt | grep '#_contigs' | grep -v 'gt_' | awk '{ print $4 }'

In [ ]:
# Mean +/- standard deviation of contig count, per haplotype.
hap1_num_mean = f'{np.mean([int(a) for a in hap1_num]):,.0f} +/- {np.std([int(a) for a in hap1_num]):,.0f}'
hap1_num_mean

In [ ]:
hap2_num_mean = f'{np.mean([int(a) for a in hap2_num]):,.0f} +/- {np.std([int(a) for a in hap2_num]):,.0f}'
hap2_num_mean

In [ ]:
# Contig N50 per haplotype, and the median across samples.
hap1_n50 = !cat full_report_map.txt | grep '^N50' | awk '{ print $3 }'
hap2_n50 = !cat full_report_map.txt | grep '^N50' | awk '{ print $4 }'

In [ ]:
hap1_n50_median = f'{np.median([int(a) for a in hap1_n50]):,.0f}'
hap1_n50_median

In [ ]:
hap2_n50_median = f'{np.median([int(a) for a in hap2_n50]):,.0f}'
hap2_n50_median

In [ ]:
hap1_n50

In [ ]:
# Check that minimap2 is installed; prints its usage.
!minimap2

In [ ]:
# Download one sample's haplotype assemblies and its GRCh38-aligned reads, for a spot-check alignment
# of reads back to the sample's own assembly.
!gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/pooled/longreads/v7_base/hifiasm/example_sample/example_sample.haploTigs/* .

In [ ]:
# Combine both haplotypes' primary contigs into one FASTA.
!zcat example_sample.bp.hap*.p_ctg.fa.gz > example_sample.bp.p_ctg.fa

In [ ]:
!minimap2

In [ ]:
!gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/pooled/longreads/v7_base/bam/example_sample/GRCh38/example_sample.bam .

In [ ]:
# Peek at the reads, then index the assembly and write the reads out as FASTA.
!samtools fastq example_sample.bam | head

In [ ]:
!samtools faidx example_sample.bp.p_ctg.fa

In [ ]:
!samtools fasta example_sample.bam > example_sample.fa

In [ ]:
!cat example_sample.fa

In [ ]:
# Align the sample's reads to its own assembly (PacBio preset, with MD tags and =/X CIGAR operations).
!minimap2 -ayYL --MD --eqx -x map-pb example_sample.bp.p_ctg.fa example_sample.fa

In [ ]:
# Index the aligned BAM.
!samtools index example_sample.aligned.bam

In [ ]:
!minimap2